# Тема 6. Агенти та інструменти

Демонстраційний код для живого заняття.

> **Версія.** Конспект і домашнє завдання цієї теми написані під **LangChain 1.x** (`create_agent`), тому ноутбук теж працює на 1.x. Старі `initialize_agent` / `AgentType` / `agent.run()` у LangChain 1.x прибрали — тут їх замінено на `create_agent` і `agent.invoke()`.

**Практичне заняття: огляд та імплементація основних інструментів LangChain**

**Мета заняття:** розібрати найпопулярніші інструменти LangChain, навчитися підключати їх до агента й аналізувати роботу агента через трейсинг.

**Що ми будемо робити:**

1. **Перший агент** — власні інструменти через `@tool` і цикл ReAct «думай → дій → спостерігай».
2. **Готові інструменти** — пошук Google (Serper), Wikipedia, погода, Python-інтерпретатор, SQL, Pandas.
3. **Агент з кількома інструментами** — модель сама обирає, що і в якому порядку викликати.
4. **Наскрізна задача** — агент Raketa Coffee з довготривалою пам'яттю (як у ДЗ 3).
5. **Трейсинг** — кожен крок агента в LangSmith.

**Базові матеріали:** [стаття з Medium](https://medium.com/@Shamimw/understanding-langchain-tools-and-agents-a-guide-to-building-smart-ai-applications-e81d200b3c12) (написана під старий API LangChain; ідеї ті самі, синтаксис — як у цьому ноутбуці).

---

Починайте урок з цієї сторінки: [docs.langchain.com/oss/python/integrations/tools](https://docs.langchain.com/oss/python/integrations/tools)

Покажіть студентам увесь обсяг готових інтеграцій LangChain, які можна підключити як інструменти.

### Встановлення залежностей

Версії зафіксовані, щоб код не зламався з наступним оновленням LangChain.

**У Google Colab:** якщо після встановлення з'явиться попередження про перезапуск, натисніть *Runtime → Restart session* і запустіть ноутбук спочатку.

In [2]:
!pip install -q "langchain==1.4.6" "langchain-core==1.6.12" "langchain-openai==1.7.2" "langchain-community==0.4.2" "langchain-experimental==0.4.2" "langchain-tavily==0.2.18" "langgraph==1.2.16" "wikipedia==1.4.0" tabulate

In [3]:
from importlib.metadata import version

for pkg in ["langchain", "langchain-core", "langchain-openai", "langchain-community", "langgraph", "langsmith"]:
    print(f"{pkg:22} {version(pkg)}")

langchain              1.4.6
langchain-core         1.6.12
langchain-openai       1.7.2
langchain-community    0.4.2
langgraph              1.2.16
langsmith              0.14.11


### Налаштування API ключів і трасування

Ключі не пишемо прямо в коді. У Colab додайте їх у **Secrets** (іконка ключа зліва) й увімкніть *Notebook access*:

| Назва | Де взяти | Обов'язковий? |
| --- | --- | --- |
| `OPENROUTER_API_KEY` | [openrouter.ai/keys](https://openrouter.ai/keys) | так |
| `LANGSMITH_API_KEY` | [smith.langchain.com](https://smith.langchain.com) → Settings → API Keys | так, для трейсингу |
| `SERPER_API_KEY` | [serper.dev](https://serper.dev) — пошук Google | лише для розділу 2 |

Якщо ключа немає в Secrets, після запуску клітинки з'явиться поле для введення. Порожній Enter — пропустити необов'язковий ключ.

Трасування вмикаємо **до першого виклику моделі**: LangChain запам'ятовує ці налаштування при першому запуску.

In [4]:
import os
from getpass import getpass

def get_secret(name: str, required: bool = True) -> str:
    # 1) змінна середовища, 2) Colab Secrets, 3) ручне введення
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    hint = "" if required else " (Enter — пропустити)"
    return getpass(f"Введіть {name}{hint}: ")

API_KEY = get_secret("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4o-mini"   # модель має підтримувати tool calling

for name in ["SERPER_API_KEY", "TAVILY_API_KEY"]:
    value = get_secret(name, required=False)
    if value:
        os.environ[name] = value

# LangSmith: усі запуски агентів потраплять у проєкт neoversity-topic-6
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = get_secret("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "neoversity-topic-6"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
# Якщо акаунт LangSmith у EU-регіоні:
# os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

Введіть TAVILY_API_KEY (Enter — пропустити): ··········


### Модель

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model=MODEL,
    temperature=0,
)

## 1. Перший агент: власні інструменти та цикл ReAct

Інструмент — це звичайна Python-функція з декоратором `@tool`. Модель бачить **назву, docstring і типи аргументів** і за ними вирішує, коли й з якими аргументами викликати інструмент.

In [28]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_price(product: str) -> str:
    """Повертає ціну товару магазину Raketa Coffee в гривнях."""
    prices = {"espresso blend": 499, "colombia huila": 239, "brazil santos": 459}
    price = prices.get(product.lower())
    return f"{product}: {price} грн" if price else f"Товар «{product}» не знайдено"

@tool
def calculate(expression: str) -> str:
    """Обчислює арифметичний вираз, наприклад '2 * 1000 + 300'."""
    return str(eval(expression, {"__builtins__": {}}))   # ⚠️ eval — лише для демо

agent = create_agent(
    llm,
    tools=[get_price, calculate],
    system_prompt="Ти асистент магазину кави. Ціни бери тільки з інструмента get_price, рахуй через calculate.",
)

print(get_price.name, "—", get_price.description)
print(get_price.args)

get_price — Повертає ціну товару магазину Raketa Coffee в гривнях.
{'product': {'title': 'Product', 'type': 'string'}}


Запускаємо агента і дивимось на всю історію повідомлень: тут видно цикл ReAct

In [7]:
result = agent.invoke({"messages": [
    {"role": "user", "content": "Скільки коштують 2 пачки Espresso Blend і 1 пачка Colombia Huila разом?"}
]})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Скільки коштують 2 пачки Espresso Blend і 1 пачка Colombia Huila разом?
================================== Ai Message ==================================
Tool Calls:
  get_price (call_0ut1MMOEDie21PvQMRec7Iro)
 Call ID: call_0ut1MMOEDie21PvQMRec7Iro
  Args:
    product: Espresso Blend
  get_price (call_Gl0QTsYplMRGhysb3owrYqCi)
 Call ID: call_Gl0QTsYplMRGhysb3owrYqCi
  Args:
    product: Colombia Huila
================================= Tool Message =================================
Name: get_price

Espresso Blend: 499 грн
================================= Tool Message =================================
Name: get_price

Colombia Huila: 239 грн
================================== Ai Message ==================================
Tool Calls:
  calculate (call_XHg3CHqlUL7EuyDGS1fYSgU3)
 Call ID: call_XHg3CHqlUL7EuyDGS1fYSgU3
  Args:
    expression: 2 * 499 + 239
================================= Tool Message =======

`Human` → `Ai` з **Tool Calls** (думка + дія) → `Tool` (спостереження) → … → `Ai` з фінальною відповіддю. Правильна відповідь: 2 × 499 + 239 = **1237 грн**.

Той самий цикл можна дивитися крок за кроком у реальному часі:

In [8]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Скільки коштує Brazil Santos?"}]},
    stream_mode="updates",
):
    for node, update in step.items():
        print(f"--- {node} ---")
        update["messages"][-1].pretty_print()

--- model ---
================================== Ai Message ==================================
Tool Calls:
  get_price (call_5g4mzfcXhtC2NYWcN32PLYCi)
 Call ID: call_5g4mzfcXhtC2NYWcN32PLYCi
  Args:
    product: Brazil Santos
--- tools ---
================================= Tool Message =================================
Name: get_price

Brazil Santos: 459 грн
--- model ---
================================== Ai Message ==================================

Ціна Brazil Santos становить 459 гривень.


## 2. Пошук Google (Serper)

[serper.dev](https://serper.dev) → зареєструватися → API Key (є безкоштовні запити для старту). Покажіть студентам, де взяти ключ.

> Зверніть увагу: `GoogleSerperAPIWrapper` працює саме з **serper.dev**, а не з serpapi.com — це різні сервіси з різними ключами.

In [9]:
from langchain_community.utilities import GoogleSerperAPIWrapper

if os.environ.get("SERPER_API_KEY"):
    search = GoogleSerperAPIWrapper()

    @tool
    def google_search(query: str) -> str:
        """Шукає актуальну інформацію в Google. Використовуй для новин, подій і фактів після дати навчання моделі."""
        return search.run(query)

    search_agent = create_agent(llm, tools=[google_search])
    result = search_agent.invoke({"messages": [{"role": "user", "content": "Tell me about LangChain. What is the latest version?"}]})
    print(result["messages"][-1].content)
else:
    print("SERPER_API_KEY не задано — розділ пропущено")

/tmp/ipykernel_11216/651210974.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


LangChain is a framework designed for building applications powered by large language models (LLMs). It provides developers with tools to create agents and applications that leverage LLM capabilities effectively.

As of now, the latest version of LangChain is **1.5.18**, which was published just 6 days ago. This version includes various improvements and updates to enhance the framework's functionality and usability.


**Варіант із конспекту — Tavily** (пошук, створений спеціально для LLM). Потрібен `TAVILY_API_KEY`.

In [ ]:
if os.environ.get("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch

    tavily = TavilySearch(max_results=5)
    tavily_agent = create_agent(
        llm,
        tools=[tavily],
        system_prompt="Ти — корисний асистент. Використовуй пошук для знаходження актуальної інформації.",
    )
    result = tavily_agent.invoke({"messages": [{"role": "user", "content": "Які останні новини про штучний інтелект?"}]})
    print(result["messages"][-1].content)
else:
    print("TAVILY_API_KEY не задано — розділ пропущено")

## 3. Wikipedia

Готовий інструмент з `langchain_community`, ключ не потрібен. - Обмежила доступ для такого роду запитів

In [10]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import os

# Налаштовуємо кастомний User-Agent для клієнта wikipedia на рівні середовища або ініціалізації
# Вікіпедії потрібен унікальний ідентифікатор у User-Agent
import wikipedia
wikipedia.set_user_agent("MyLangChainAgent/1.0 (contact: test@example.com)")

try:
    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=2000))
    print(wiki.name, "—", wiki.description[:120], "...")

    wiki_agent = create_agent(llm, tools=[wiki])
    result = wiki_agent.invoke({"messages": [{"role": "user", "content": "In 2 lines tell me about Italy"}]})
    print(result["messages"][-1].content)
except Exception as e:
    print(f"Не вдалося виконати запит до Вікіпедії через помилку: {e}")

wikipedia — A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, ...
Italy, officially the Italian Republic, is a country in Southern and Western Europe known for its rich history, culture, and cuisine. It features a peninsula extending into the Mediterranean Sea and is home to major cities like Rome, Milan, and Venice.


## 4. Погода (власний інструмент над зовнішнім API)

Використовуємо [Open-Meteo](https://open-meteo.com) — безкоштовний API без ключа. Інструмент робить два запити: місто → координати, координати → поточна погода.

In [23]:
import requests

@tool
def get_weather(city: str) -> str:
    """Повертає поточну погоду в місті: температуру (°C), вітер (км/год) і вологість (%)."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}, timeout=10,
    ).json()
    if not geo.get("results"):
        return f"Місто «{city}» не знайдено"          # безпечне відмовлення замість винятку
    place = geo["results"][0]
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": place["latitude"], "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,relative_humidity_2m"},
        timeout=10,
    ).json()["current"]
    return (f"{place['name']}: {weather['temperature_2m']} °C, вітер {weather['wind_speed_10m']} км/год, "
            f"вологість {weather['relative_humidity_2m']}%")

# Інструмент можна викликати й напряму — так зручно його тестувати
print(get_weather.invoke({"city": "Kyiv"}))

Kyiv: 11.6 °C, вітер 10.3 км/год, вологість 93%


In [12]:
weather_agent = create_agent(llm, tools=[get_weather])
result = weather_agent.invoke({"messages": [{"role": "user", "content": "Чи потрібна парасолька і куртка у Львові зараз?"}]})
print(result["messages"][-1].content)

У Львові зараз температура 12.4 °C, вітер 11.9 км/год, а вологість 95%. Це свідчить про те, що погода досить прохолодна і волога. Рекомендується взяти куртку, а також парасольку, оскільки висока вологість може свідчити про можливий дощ.


## 5. Python-інтерпретатор

Агент пише код і сам його виконує — так він точно рахує, а не «вгадує» (обчислювальні інструменти з конспекту).

> ⚠️ Інструмент виконує будь-який код без обмежень. У продакшні — лише в ізольованому середовищі (sandbox).

In [13]:
from langchain_experimental.tools import PythonREPLTool

python_tool = PythonREPLTool()

# Покращуємо інструкцію, щоб модель писала простий та надійний код для обчислень
python_agent = create_agent(
    llm,
    tools=[python_tool],
    system_prompt=(
        "Ти — Python-помічник. Твоє завдання — писати та запускати чистий Python-код за допомогою інструменту Python_REPL. "
        "Обов'язково використовуй функцію print() у своєму коді, щоб вивести кінцевий результат обчислень."
    ),
)

try:
    result = python_agent.invoke({"messages": [{"role": "user", "content": "Скільки днів між 1 вересня 2026 і 31 грудня 2026? І скільки з них — неділі?"}]})
    for message in result["messages"]:
        message.pretty_print()
except Exception as e:
    print(f"Помилка виконання: {e}")

/tmp/ipykernel_11216/3014278625.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool


================================ Human Message =================================

Скільки днів між 1 вересня 2026 і 31 грудня 2026? І скільки з них — неділі?
================================== Ai Message ==================================
Tool Calls:
  Python_REPL (call_jgzlSuMeV8JtJ1qQlGFQZNxT)
 Call ID: call_jgzlSuMeV8JtJ1qQlGFQZNxT
  Args:
    query: from datetime import datetime, timedelta

# Визначаємо дати
start_date = datetime(2026, 9, 1)
end_date = datetime(2026, 12, 31)

# Обчислюємо кількість днів
delta = end_date - start_date
num_days = delta.days

# Обчислюємо кількість неділь
num_sundays = sum(1 for i in range(num_days + 1) if (start_date + timedelta(days=i)).weekday() == 6)

num_days, num_sundays
  Python_REPL (call_agzphjNWwPqC56DHPxtDvcjS)
 Call ID: call_agzphjNWwPqC56DHPxtDvcjS
  Args:
    query: from datetime import datetime, timedelta

# Визначаємо дати
start_date = datetime(2026, 9, 1)
end_date = datetime(2026, 12, 31)

# Обчислюємо кількість днів
delta = end_date -

Правильна відповідь: 121 день між датами (122, якщо рахувати обидві дати включно), з них **17 неділь**.

## 6. SQL-база (не обов'язково показувати)

Замість MySQL беремо вбудовану SQLite — вона працює в Colab без сервера. Інструмент дозволяє лише `SELECT`: приклад принципу безпечного відмовлення з конспекту.

In [18]:
import sqlite3
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

# Тестова база в пам'яті
engine = create_engine("sqlite://", poolclass=StaticPool, connect_args={"check_same_thread": False})
with engine.begin() as conn:
    conn.exec_driver_sql("CREATE TABLE orders (id INTEGER, product TEXT, qty INTEGER, price INTEGER, city TEXT)")
    conn.exec_driver_sql("""INSERT INTO orders VALUES
        (1, 'Espresso Blend', 2, 499, 'Київ'), (2, 'Colombia Huila', 1, 239, 'Львів'),
        (3, 'Espresso Blend', 1, 499, 'Одеса'), (4, 'Brazil Santos', 3, 459, 'Київ'),
        (5, 'Colombia Huila', 4, 239, 'Київ')""")

db = SQLDatabase(engine)

@tool
def sql_query(query: str) -> str:
    """Виконує SELECT-запит до бази магазину.Таблиця orders(id, product, qty, price, city)."""
    if not query.strip().lower().startswith("select"):
        return "Дозволені лише SELECT-запити"
    return db.run(query)

print(sql_query.invoke({"query": "SELECT COUNT(*) FROM orders"}))

sql_agent = create_agent(llm, tools=[sql_query])
result = sql_agent.invoke({"messages": [{"role": "user", "content": "Який товар приніс найбільшу виручку і скільки?"}]})
print(result["messages"][-1].content)

[(5,)]
На жаль, я можу виконувати лише SELECT-запити до бази даних. Якщо вам потрібно видалити таблицю, вам слід зробити це безпосередньо через систему управління базами даних, яку ви використовуєте. Якщо у вас є інші запити або питання, я з радістю допоможу!


Правильна відповідь: **Espresso Blend — 1497 грн** (3 пачки × 499). Далі Brazil Santos — 1377 грн, Colombia Huila — 1195 грн. У LangSmith покажіть, який SQL-запит написала модель.

## 7. Pandas DataFrame (обов'язково показати)

Агент відповідає на питання про таблицю, сам пишучи pandas-код.

In [19]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent

df = pd.DataFrame({
    "name":    ["Олена", "Андрій", "Марія", "Тарас", "Ірина", "Богдан"],
    "math":    [92, 78, 88, 95, 67, 81],
    "english": [85, 90, 72, 70, 94, 77],
    "python":  [98, 65, 91, 88, 73, 84],
})
df

,name,math,english,python
0,Олена,92,85,98
1,Андрій,78,90,65
2,Марія,88,72,91
3,Тарас,95,70,88
4,Ірина,67,94,73
5,Богдан,81,77,84


In [21]:
pandas_agent = create_pandas_dataframe_agent(
    llm,
    df,
    agent_type="tool-calling",
    verbose=True,
    allow_dangerous_code=True,   # агент виконує згенерований pandas-код
)

response = pandas_agent.invoke({"input": "У кого найбільша різниця між python і english?"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['difference'] = df['python'] - df['english']\nmax_difference_index = df['difference'].idxmax()\nresult = df.loc[max_difference_index, ['name', 'difference']]\nresult"}`


name          Марія
difference       19
Name: 2, dtype: objectУ Марії найбільша різниця між оцінками з Python і English, яка становить 19 балів.

> Finished chain.
У Марії найбільша різниця між оцінками з Python і English, яка становить 19 балів.


Правильна відповідь: найвищий бал з math — **Тарас (95)**; найкращий середній бал — **Олена (91,7)**.

## 8. Агент з кількома інструментами

Даємо агенту одразу кілька інструментів і запит, для якого потрібні два з них. Модель сама вирішує, які викликати і в якому порядку — подивіться на це в LangSmith.

In [24]:
multi_agent = create_agent(
    llm,
    tools=[get_weather, wiki, python_tool, get_price, calculate],
    system_prompt="Ти корисний асистент. Використовуй інструменти, коли потрібні факти або обчислення.",
)

result = multi_agent.invoke({"messages": [{"role": "user", "content":
    "Яка зараз температура в Римі і в Києві? Обчисли різницю між ними."}]})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Яка зараз температура в Римі і в Києві? Обчисли різницю між ними.
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_x9nBKHMY7mKvQrYCME3kiRxd)
 Call ID: call_x9nBKHMY7mKvQrYCME3kiRxd
  Args:
    city: Рим
  get_weather (call_CQR38p9hCvw17se7dbCo35Gt)
 Call ID: call_CQR38p9hCvw17se7dbCo35Gt
  Args:
    city: Київ
================================= Tool Message =================================
Name: get_weather

Rimsko-Korsakovka: 6.3 °C, вітер 6.0 км/год, вологість 64%
================================= Tool Message =================================
Name: get_weather

Місто «Київ» не знайдено
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_9xcFFg5t2X522UKACsklRcR7)
 Call ID: call_9xcFFg5t2X522UKACsklRcR7
  Args:
    city: Київ
================================= Tool Message =========

## 9. Наскрізна задача: агент Raketa Coffee з довготривалою пам'яттю

Збираємо в одного агента інструменти з цього заняття (ціни, калькулятор, база замовлень) і додаємо **довготривалу пам'ять** через `InMemoryStore`. Кожен `invoke` — нова розмова без історії, але вподобання клієнта зберігаються між ними. Саме так працює ДЗ 3 (TODO-агент).

In [25]:
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store

store = InMemoryStore()

@tool
def save_preference(customer: str, preference: str) -> str:
    """Зберігає вподобання клієнта (улюблений сорт, помел) у довготривалу пам'ять."""
    get_store().put(("customers",), customer.lower(), {"preference": preference})
    return f"Збережено для {customer}: {preference}"

@tool
def get_preference(customer: str) -> str:
    """Повертає збережені вподобання клієнта за іменем."""
    item = get_store().get(("customers",), customer.lower())
    return item.value["preference"] if item else f"Для {customer} нічого не збережено"

raketa_agent = create_agent(
    llm,
    tools=[get_price, calculate, sql_query, save_preference, get_preference],
    store=store,
    system_prompt=(
        "Ти асистент магазину кави Raketa Coffee. Ціни бери з get_price, рахуй через calculate, "
        "статистику замовлень — через sql_query. Коли клієнт називає свої вподобання, збережи їх; "
        "коли клієнт представився, перевір збережені вподобання."
    ),
)

Сесія 1: клієнт розповідає про себе

In [26]:
r1 = raketa_agent.invoke({"messages": [{"role": "user", "content":
    "Я Олена. Запам'ятай, будь ласка: я завжди беру Colombia Huila, помел під фільтр."}]})
print(r1["messages"][-1].content)

Я запам'ятав твої вподобання, Олено: ти завжди береш Colombia Huila з помелом під фільтр. Чи можу я чимось ще допомогти?


Сесія 2: новий виклик — історії розмови немає, але пам'ять є

In [29]:
r2 = raketa_agent.invoke({"messages": [{"role": "user", "content":
    "Привіт, це Олена. Що я зазвичай беру і скільки коштуватимуть 3 пачки?"}]})
for message in r2["messages"]:
    message.pretty_print()

print("\nУ сховищі:", [(item.key, item.value) for item in store.search(("customers",))])

================================ Human Message =================================

Привіт, це Олена. Що я зазвичай беру і скільки коштуватимуть 3 пачки?
================================== Ai Message ==================================
Tool Calls:
  get_preference (call_wk7H11w7zwZm18gcj9z9dl2k)
 Call ID: call_wk7H11w7zwZm18gcj9z9dl2k
  Args:
    customer: Олена
================================= Tool Message =================================
Name: get_preference

Colombia Huila, помел під фільтр
================================== Ai Message ==================================
Tool Calls:
  get_price (call_dONBi6ig5J1uwpEQwfA3w05f)
 Call ID: call_dONBi6ig5J1uwpEQwfA3w05f
  Args:
    product: Colombia Huila
  get_price (call_WvXZB4H6fXq4qOYSlQwDyG4y)
 Call ID: call_WvXZB4H6fXq4qOYSlQwDyG4y
  Args:
    product: Colombia Huila
================================= Tool Message =================================
Name: get_price

Colombia Huila: 239 грн
================================= Tool Message 

Очікувано: агент викликає `get_preference("Олена")`, потім `get_price` і `calculate` → 3 × 239 = **717 грн**.

## 10. Трейсинг у LangSmith

Відкрийте [smith.langchain.com](https://smith.langchain.com) → проєкт **neoversity-topic-6**. Для кожного запуску агента видно:

- дерево кроків: `model` → `tools` → `model` → … (той самий цикл ReAct);
- які інструменти й з якими аргументами викликала модель;
- що повернув кожен інструмент;
- час і токени кожного кроку.

Запуск нижче має власну назву й теги — його легко знайти фільтром.

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Порахуй вартість 3 пачок Brazil Santos"}]},
    config={"run_name": "coffee-agent-demo", "tags": ["topic-6", "react"]},
)
print(result["messages"][-1].content)

wait_for_all_tracers()   # чекаємо, поки трейси відправляться
print("Готово — відкрийте проєкт neoversity-topic-6 у LangSmith")

In [30]:
os.environ["LANGSMITH_TRACING"] = "false"
ls_utils.get_env_var.cache_clear()

NameError: name 'ls_utils' is not defined

---

## Підсумок

- **Агент** = модель + інструменти + цикл «думай → дій → спостерігай» (`create_agent`).
- **Інструмент** = функція з `@tool`: назва, docstring і типи — це «інструкція» для моделі.
- **Готові інтеграції**: пошук (Serper, Tavily), Wikipedia, Python, SQL, Pandas — і сотні інших на сторінці інтеграцій.
- **Пам'ять**: історія діалогу живе в межах одного виклику, а `InMemoryStore` зберігає дані між викликами.
- **LangSmith** показує, які інструменти, у якому порядку і з якими аргументами викликав агент.